# Pandas API on Spark (pyspark.pandas)
La **API de Pandas sobre Spark** permite trabajar con datos distribuidos usando una sintaxis similar a Pandas.
Esto combina la **facilidad de Pandas** con la **escalabilidad de Spark**.

## ✅ Características principales:
- Sintaxis similar a Pandas.
- Ejecución distribuida sobre Spark.
- Permite `.map()`, `.apply()`, `groupby()`, `merge()`, etc.

## ✅ Diferencias con DataFrames tradicionales de Spark:
| Operación | Spark DataFrame | Pandas API on Spark |
|-----------|-----------------|---------------------|
| Selección | `df.select('col')` | `psdf[['col']]` |
| Filtro    | `df.filter(col > 5)` | `psdf[psdf.col > 5]` |
| Nueva columna | `df.withColumn('x', col*2)` | `psdf['x'] = psdf.col * 2` |
| Función personalizada | UDF | `.apply()` |


In [ ]:

import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import math

# Crear sesión Spark
spark = SparkSession.builder.appName("PandasAPIonSparkDemo").getOrCreate()

# Definir esquema
schema = StructType([
    StructField("codigo_", StringType(), True),
    StructField("TIPO", StringType(), True),
    StructField("EJE", StringType(), True),
    StructField("NOMBRE_DEL_OBJETIVO", StringType(), True),
    StructField("NOMBRE_DE_LA_POLITICA", StringType(), True),
    StructField("META", StringType(), True),
    StructField("INDICADOR", StringType(), True),
    StructField("FUENTE_DE_INFORMACION", StringType(), True),
    StructField("GRUPO_DE_DESAGREGACION", StringType(), True),
    StructField("NIVEL_DE_DESAGREGACION", StringType(), True),
    StructField("CODIGO_GEOGRAFICO_DPA", StringType(), True),
    StructField("MES_ANIO", StringType(), True),
    StructField("FECHA", DateType(), True),
    StructField("ESTIMADOR", DoubleType(), True),
    StructField("ERROR_ESTANDAR", DoubleType(), True),
    StructField("LIMITE_INFERIOR", DoubleType(), True),
    StructField("LIMITE_SUPERIOR", DoubleType(), True),
    StructField("COEFICIENTE_DE_VARIACION", DoubleType(), True),
    StructField("NUMERADOR", DoubleType(), True),
    StructField("DENOMINADOR", DoubleType(), True),
    StructField("NOMBRE_DEL_EJE", StringType(), True),
    StructField("PERIODICIDAD_FICHA_METODOLOGICA", StringType(), True),
    StructField("FECHA_DE_TRANSFERENCIA_FICHA_METODOLOGICA", StringType(), True),
    StructField("DESAGREGACION_FICHA_METODOLOGICA", StringType(), True),
    StructField("PERIODICIDAD_DEL_DATO", StringType(), True)
])

# Simulación de datos (usando un pequeño dataset para ejemplo)
data = [
    ("A1", "Tipo1", "Eje1", "Objetivo1", "Politica1", "Meta1", "Ind1", "Fuente1", "Grupo1", "Nivel1", "DPA1", "2025-05", None, 120.0, 1.5, 100.0, 140.0, 0.1, 500.0, 1000.0, "EjeName1", "Mensual", "2025-05-01", "Desag1", "Mensual"),
    ("B2", "Tipo2", "Eje2", "Objetivo2", "Politica2", "Meta2", "Ind2", "Fuente2", "Grupo2", "Nivel2", "DPA2", "2025-06", None, 80.0, 2.0, 60.0, 100.0, 0.2, 400.0, 800.0, "EjeName2", "Trimestral", "2025-06-01", "Desag2", "Trimestral")
]
df = spark.createDataFrame(data, schema=schema)

# Convertir a Pandas API on Spark
psdf = df.pandas_api()
psdf.head(3)


## 1. Ejemplos de `.map()` en Pandas API on Spark
La función `.map()` se aplica sobre Series (`psdf['col']`) y permite aplicar operaciones elemento por elemento.


In [ ]:

# Ejemplo 1: duplicar valores
psdf['ESTIMADOR_X2'] = psdf['ESTIMADOR'].map(lambda x: x * 2 if x is not None else None)

# Ejemplo 2: clasificar valores
psdf['CLASE'] = psdf['ESTIMADOR'].map(lambda x: 'ALTO' if x and x > 100 else 'BAJO')

# Ejemplo 3: raíz cuadrada segura
psdf['SQRT_ESTIMADOR'] = psdf['ESTIMADOR'].map(lambda x: math.sqrt(x) if x is not None and x >= 0 else None)

# Ejemplo 4: inicial de EJE
psdf['INICIAL_EJE'] = psdf['EJE'].map(lambda x: x[0] if x else '?')

# Ejemplo 5: concatenar codigo y eje
psdf['CODIGO_FULL'] = psdf.apply(lambda row: f"{row.codigo_}-{row.EJE}" if row.codigo_ and row.EJE else None, axis=1)

psdf[['ESTIMADOR','ESTIMADOR_X2','CLASE','SQRT_ESTIMADOR','CODIGO_FULL']].head(5)


## 2. Operaciones comunes en Pandas API on Spark
- **Filtrado**: igual que en Pandas, usando condiciones.
- **Agrupaciones**: `.groupby()` y agregaciones.
- **Ordenamientos**: `.sort_values()`.


In [ ]:

# Filtrar registros con ESTIMADOR > 50
psdf_filtrado = psdf[psdf['ESTIMADOR'] > 50]

# Agrupar y calcular promedio
psdf_group = psdf.groupby('EJE')['ESTIMADOR'].mean()

# Ordenar por ESTIMADOR
psdf_sorted = psdf.sort_values('ESTIMADOR', ascending=False)

print(psdf_filtrado.head(3))
print(psdf_group.head(3))
print(psdf_sorted.head(3))


## ✅ Conclusión
- La **API de Pandas sobre Spark** permite escribir código estilo Pandas, pero ejecutado de manera distribuida.
- `.map()` y `.apply()` funcionan igual que en Pandas, pero sobre grandes datasets.
- Es ideal para quienes conocen Pandas y quieren escalar sin reescribir lógica a DataFrames Spark.


In [ ]:
spark.stop()